# TCP Congestion Anomaly Detection via LSTM

This notebook trains a 2-layer LSTM off of 255 TCP runs of data.

## Setup

In [ ]:
%pip install tensorflow pandas numpy scikit-learn matplotlib --quiet

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS_DIR       = '/content/drive/MyDrive/utcp/runs'
    CACHE_DIR      = '/content/drive/MyDrive/utcp/cache_kernel_v1'
    MODEL_SAVE_DIR = '/content/drive/MyDrive/utcp/models'
    IN_COLAB = True
except ImportError:
    RUNS_DIR       = '../log/runs'
    CACHE_DIR      = '../log/cache_kernel_v1'
    MODEL_SAVE_DIR = '../log/models'
    IN_COLAB = False

print(f'Running in {"Colab" if IN_COLAB else "local"} mode')
print(f'RUNS_DIR = {RUNS_DIR}')

In [ ]:
import os, random, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, confusion_matrix, f1_score,
    classification_report, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
SEED = 15
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f'TensorFlow {tf.__version__}')

## Configuration

In [ ]:
'''
Data config and labeling
'''
BIN_SIZE_MS = 15 # amount of time that makes up e/a bin (in ms)

N_MIN_BINS = 10 # minimum pre-onset bins (10 x 15 ms = 150 ms)
N_PRE_ONSET_RTTS = 2 # RTTs before each onset to label

# lstm_output.csv's raw columns
RAW_COLS = [
    'zlog_ts', 'timestamp_us', 'rtt_us', 'srtt_us', 'rttvar_us', 'rto_us',
    'min_rtt_us', 'queue_delay_us', 'rtt_delta_us', 'rtt_accel_us',
    'rto_delta_us', 'cwnd', 'ssthresh', 'snd_wnd', 'flight_size', 'newly_acked',
    'inter_ack_us', 'ca_state', 't_dupacks', 't_rxtshift', 'is_dup_ack',
    'is_timeout',
]

# features to train the LSTM on
FEATURE_COLS = [
    'srtt_us',          # smoothed RTT
    'rttvar_us',        # RTT variance
    'queue_delay_us',   # rtt - min_rtt
    'rtt_delta_us',     # d(RTT)/dt
    'rtt_accel_us',     # d^2(RTT)/dt^2
    'utilization',      # flight_size / effective_window
    'threshold_ratio',  # RTT / (SRTT + K * RTTVAR)
]

N_FEATURES = len(FEATURE_COLS)

K = 2 # value for threshold_ratio

'''
LSTM config and architecture
'''
SEQ_LEN = 200 # bins per input window (200 bins x 15ms = 3s lookback window)
STRIDE  = 3   # sliding-window step

# architecture-related
HIDDEN_UNITS  = 64
DROPOUT_LSTM  = 0.3
DROPOUT_DENSE = 0.2

# training-related
BATCH_SIZE = 128
EPOCHS     = 50
PATIENCE   = 8
LR         = 1e-3

# ratios for data splitting
TRAIN_FRAC = 0.60
VAL_FRAC   = 0.20 # final 20% is used for testing

# paths for saving results metadata
MODEL_SAVE = 'rtt_kernel_lstm_v1.keras'
META_SAVE  = 'rtt_kernel_lstm_v1_meta.json'
model_path = os.path.join(MODEL_SAVE_DIR, MODEL_SAVE)
meta_path  = os.path.join(MODEL_SAVE_DIR, META_SAVE)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f'Config loaded: There are {N_FEATURES} features, {BIN_SIZE_MS} ms bins, K={K}')

## Data Loading & Binning

Each run is loaded, normalized by `min_rtt`, and binned into 15 ms windows. The `utilization` feature is computed for each row, and the `threshold_ratio` feature is computed for each bin.

In [ ]:
def load_and_bin_run(path: str) -> tuple[pd.DataFrame, float] | None:
    """
    Load one lstm_output.csv file, normalize the data, bin it into 15
    ms windows, and compute features.

    Returns `(binned_df, run_min_rtt_us)` or `None` on failure.

    - `binned_df` contains `FEATURE_COLS` plus 'ca_state' and 'is_timeout'
    for kernel-based labeling.
    """
    try:
        df = pd.read_csv(
            path, names=RAW_COLS, header=None,
            on_bad_lines='skip', low_memory=False
        )
        df = df.apply(pd.to_numeric, errors='coerce')
        df.drop(columns=['zlog_ts'], inplace=True)
        df.dropna(subset=['timestamp_us', 'min_rtt_us', 'srtt_us'], inplace=True)
        df = df[df['min_rtt_us'] > 0].reset_index(drop=True)

        if len(df) < 100:
            return None

        '''
        Normalize each row according to the median of the run's min RTT
        '''
        run_min_rtt = df['min_rtt_us'].replace(0, np.nan).median()
        if pd.isna(run_min_rtt) or run_min_rtt <= 0:
            run_min_rtt = 1.0 # fallback value

        # normalize RTT-related columns
        for col in ['srtt_us', 'rttvar_us', 'queue_delay_us', 'rtt_delta_us', 'rtt_accel_us']:
            df[col] = df[col] / run_min_rtt

        # Normalize the RTT and store it in a new col for the threshold_ratio
        df['rtt_us_norm'] = df['rtt_us'] / run_min_rtt

        # Compute the utilization
        eff_win = df[['cwnd', 'snd_wnd']].min(axis=1).clip(lower=1)
        df['utilization'] = (df['flight_size'] / eff_win).clip(0.0, 2.0)

        '''
        Put the normalized rows into 15ms bins
        '''
        df['rel_time_ms'] = (df['timestamp_us'] - df['timestamp_us'].iloc[0]) / 1000.0
        max_time = df['rel_time_ms'].max()
        bin_edges = np.arange(0, max_time + BIN_SIZE_MS, BIN_SIZE_MS)

        # right=False means bins are half-open on the right (e.g., [0, 15), [15, 30))
        df['bin'] = pd.cut(df['rel_time_ms'], bins=bin_edges, labels=False, right=False)

        df.dropna(subset=['bin'], inplace=True)
        df['bin'] = df['bin'].astype(int)

        # Collapse all of the rows in a bin into a single row, either by avg value or max value
        agg = {
            'srtt_us':        'mean',
            'rttvar_us':      'mean',
            'queue_delay_us': 'max',
            'rtt_delta_us':   'mean',
            'rtt_accel_us':   'mean',
            'utilization':    'mean',
            'rtt_us_norm':    'mean',
            # The rest are for labeling, not features
            'ca_state':       'max',
            'is_timeout':     'max',
            'cwnd':           'mean',
        }
        binned = df.groupby('bin').agg(agg)

        # Fill empty bins with linear interpolation
        all_bins = pd.RangeIndex(binned.index.min(), binned.index.max() + 1)
        binned = binned.reindex(all_bins).interpolate(method='linear').fillna(0)

        # The CSV must produce at least 201 bins, else return None
        if len(binned) < SEQ_LEN + 1:
            return None

        # Compute the threshold_ratio of each bin and store it in a new column
        threshold = (binned['srtt_us'] + K * binned['rttvar_us']).clip(lower=1e-6)
        binned['threshold_ratio'] = (binned['rtt_us_norm'] / threshold).clip(0, 5)

        # Remove any extreme outliers that could mess up the LSTM's training
        for col in FEATURE_COLS:
            binned[col] = binned[col].clip(-10.0, 10.0)

        # log any exceedingly fast ( < 100 micro sec) and/or slow (> 5 sec) RTTs
        min_rtt_ms = run_min_rtt / 1000.0
        if min_rtt_ms < 0.1 or min_rtt_ms > 5000.0:
            print(f'\t[WARN] {path}: unusual run_min_rtt={min_rtt_ms:.2f} ms')

        return binned, run_min_rtt

    except Exception as e:
        print(f'\t[SKIP] {path}: {e}')
        return None


print('load_and_bin_run() defined.')

## Pre-Onset Congestion Labeling

There is congestion onset if `ca_state >= 3` or `is_timeout >= 1`.

Labels are placed in the `N-RTT` window before each onset to train the model to recognize when congestion is to begin to occur.

In [ ]:
def label_pre_onset_kernel(binned: pd.DataFrame, run_min_rtt_us: float) -> np.ndarray:
    """
    Label the N_PRE_ONSET_RTTS-RTT window BEFORE each kernel-detected
    congestion episode. Does NOT label during congestion itself.
    """
    y = np.zeros(len(binned), dtype=np.float32)

    ca_arr  = binned['ca_state'].values
    to_arr  = binned['is_timeout'].values

    # To determine how many bins to normalize,
    # SRTT needs to be denormalized back to millisec
    srtt_arr = binned['srtt_us'].values

    # Get raw SRTT val in millisec for bin-count calculation
    srtt_raw_ms = srtt_arr * run_min_rtt_us / 1000.0

    is_cong = (ca_arr >= 3) | (to_arr >= 1) # bin is for congestion event if `True`

    in_cong = False
    for k in range(len(is_cong)):
        if is_cong[k] and not in_cong:
            # Use the bin before the congestion event to avoid reading the
            # already-elevated value at the event itself.
            pre_k = max(0, k - 1)
            srtt_ms_pre = srtt_raw_ms[pre_k]

            n_bins = max(N_MIN_BINS, int((srtt_ms_pre / BIN_SIZE_MS) * N_PRE_ONSET_RTTS))
            start = max(0, k - n_bins)
            y[start:k] = 1.0  # before onset only
            in_cong = True
        elif not is_cong[k]:
            in_cong = False

    return y


print('label_pre_onset_kernel() defined.')

## Discover CSV Runs

In [ ]:
csv_paths = sorted(glob.glob(os.path.join(RUNS_DIR, '*', 'lstm_output.csv')))
assert len(csv_paths) > 0, (
    f'No lstm_output.csv files found under {RUNS_DIR}.\n'
    'Check RUNS_DIR path.'
)
print(f'Found {len(csv_paths)} runs in {RUNS_DIR}')

## Dataset Assembly & Train/Val/Test Split

Per-run caching, stride-3 sliding windows (skipping congestion-overlapping windows), and shuffled run-level 60/20/20 split.

In [ ]:
os.makedirs(CACHE_DIR, exist_ok=True)

all_X, all_y, skipped = [], [], 0
positive_counts, total_counts = [], []
valid_paths = []

for path in csv_paths:
    run_id  = os.path.basename(os.path.dirname(path))
    cache_X = os.path.join(CACHE_DIR, f'{run_id}_X.npy')
    cache_y = os.path.join(CACHE_DIR, f'{run_id}_y.npy')

    if os.path.exists(cache_X) and os.path.exists(cache_y):
        X_run = np.load(cache_X)
        y_run = np.load(cache_y)
    else:
        result = load_and_bin_run(path)
        if result is None:
            skipped += 1
            continue
        binned, run_min_rtt = result

        y_bins = label_pre_onset_kernel(binned, run_min_rtt)
        X_mat  = binned[FEATURE_COLS].values.astype(np.float32)

        # Congestion mask: skip windows overlapping active congestion
        is_cong_bin = ((binned['ca_state'].values >= 3) |
                       (binned['is_timeout'].values >= 1))

        # Sliding windows
        n = len(X_mat)
        X_list, y_list = [], []
        for s in range(0, n - SEQ_LEN, STRIDE):
            if is_cong_bin[s : s + SEQ_LEN].any():
                continue
            X_list.append(X_mat[s : s + SEQ_LEN])
            y_list.append(y_bins[s + SEQ_LEN])

        if len(X_list) == 0:
            skipped += 1
            continue

        X_run = np.array(X_list, dtype=np.float32)
        y_run = np.array(y_list, dtype=np.float32)

        np.save(cache_X, X_run)
        np.save(cache_y, y_run)

    all_X.append(X_run)
    all_y.append(y_run)
    valid_paths.append(path)
    positive_counts.append(y_run.sum())
    total_counts.append(len(y_run))

print(f'Loaded {len(all_X)} runs, skipped {skipped}')
overall_pos_rate = sum(positive_counts) / max(sum(total_counts), 1)
print(f'Overall positive rate: {overall_pos_rate:.2%}  '
      f'(windows: {sum(total_counts):,} total, {int(sum(positive_counts)):,} positive)')

In [ ]:
# Separate the run data into a 60/20/20 split for training, validation, and testing
n_runs = len(all_X)

n_test  = max(1, round(n_runs * (1.0 - TRAIN_FRAC - VAL_FRAC)))
n_val   = max(1, round(n_runs * VAL_FRAC))
n_train = n_runs - n_test - n_val

assert n_train >= 2 and n_val >= 1 and n_test >= 1, (
    f'Not enough runs: {n_runs} → train={n_train}, val={n_val}, test={n_test}'
)

print(f'Split ({n_runs} runs): train={n_train}, val={n_val}, test={n_test}')

run_indices = list(range(n_runs))
random.shuffle(run_indices)

test_idx  = run_indices[:n_test]
val_idx   = run_indices[n_test : n_test + n_val]
train_idx = run_indices[n_test + n_val:]

def concat_split(indices):
    xs = [all_X[i] for i in indices]
    ys = [all_y[i] for i in indices]
    return np.concatenate(xs, axis=0), np.concatenate(ys, axis=0)

X_train, y_train = concat_split(train_idx)
X_val,   y_val   = concat_split(val_idx)
X_test,  y_test  = concat_split(test_idx)

print(f'Train: {X_train.shape}  pos={y_train.mean():.2%}')
print(f'Val  : {X_val.shape}    pos={y_val.mean():.2%}')
print(f'Test : {X_test.shape}   pos={y_test.mean():.2%}')

## Sanity Plots

Selects 3 random runs, and for each run creates the following graphs:

- The run's normalized RTTs, congestion events, and pre-congestion labels.
- The threshold ratio for each bin in the run.
- The utilization throughout the run.

In [ ]:
from pathlib import Path

N_SANITY = min(3, len(csv_paths)) # number of runs to make graphs for
sample_paths = random.sample(csv_paths, N_SANITY)

fig, axes = plt.subplots(N_SANITY, 3, figsize=(18, 5 * N_SANITY))
if N_SANITY == 1:
    axes = axes[np.newaxis, :]

for row, path in enumerate(sample_paths):
    result = load_and_bin_run(path)
    if result is None:
        continue
    binned, run_min_rtt = result
    y_label = label_pre_onset_kernel(binned, run_min_rtt)
    t = np.arange(len(binned)) * BIN_SIZE_MS / 1000.0
    rtt_vals = binned['rtt_us_norm'].values
    cong_mask = (binned['ca_state'].values >= 3) | (binned['is_timeout'].values >= 1)

    timeout_bins = np.where(binned['is_timeout'].values >= 1)[0]

    triple_dup_bins = np.where(
        (binned['ca_state'].values >= 3) &
        (np.concatenate([[False], binned['ca_state'].values[:-1] < 3]))
    )[0]

    # Left plots: RTTs, congestion events, and pre-congestion bin labels
    ax = axes[row, 0]
    ax.plot(t, rtt_vals, linewidth=0.8, label='RTT', color='royalblue')
    y_top = np.nanmax(rtt_vals) * 1.3

    ax.fill_between(t, 0, y_top, where=y_label.astype(bool),
                    color='red', alpha=0.30, label='pre-onset congestion')

    ax.fill_between(t, 0, y_top, where=cong_mask, color='cyan',
                    alpha=0.30, label='congestion')

    if len(timeout_bins):
        ax.scatter(t[timeout_bins], rtt_vals[timeout_bins],
                   color='#d62728', marker='X', s=50, label='Timeout', zorder=5)

    if len(triple_dup_bins):
        ax.scatter(t[triple_dup_bins], rtt_vals[triple_dup_bins],
                   color='#ff7f0e', marker='o', s=50, edgecolors='black',
                   label='3x Dupe ACK', zorder=5)

    ax.set_ylim(0, y_top)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Normalized RTT')
    ax.set_title(f'Random Run {row+1}: RTT, Congestion Events, and Pre-Onset Congestion Labels')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.3)

    # Middle plots: Threshold ratios throughout the bins
    ax = axes[row, 1]
    tr = binned['threshold_ratio'].values
    ax.plot(t, tr, linewidth=0.8, color='teal', label='threshold ratio')

    ax.axhline(1.0, color='red', linestyle='--', linewidth=1, label='ratio = 1.0')

    ax.fill_between(t, 0, tr.max() * 1.2, where=y_label.astype(bool),
                    color='red', alpha=0.30, label='pre-onset congestion')

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('RTT / (SRTT + K · RTTVAR)')
    ax.set_title(f'Random Run {row+1}: Threshold Ratio')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Right plot: utilization throughout the runs
    ax = axes[row, 2]
    util = binned['utilization'].values
    ax.plot(t, util, linewidth=0.8, color='orchid', label='utilization')

    ax.fill_between(t, 0, 2.0, where=y_label.astype(bool),
                    color='red', alpha=0.30, label='pre-onset congestion')

    ax.set_ylim(0, 2.0)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Bytes in Flight / min(cwnd, snd_wnd)')
    ax.set_title(f'Random Run {row+1}: Utilization')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sanity_plots.png', dpi=100)
plt.show()

## Model Definition

In [ ]:
tf.random.set_seed(SEED)

def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt      = tf.where(tf.equal(y_true, 1.0), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1.0), alpha, 1 - alpha)
        return -tf.reduce_mean(alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss

model = Sequential([
    LSTM(128, input_shape=(SEQ_LEN, N_FEATURES),
         return_sequences=True, name='lstm_1'),
    Dropout(DROPOUT_LSTM, name='drop_lstm_1'),
    LSTM(HIDDEN_UNITS, return_sequences=False, name='lstm_2'),
    Dropout(DROPOUT_LSTM, name='drop_lstm_2'),
    Dense(32, activation='relu', name='dense_1'),
    Dropout(DROPOUT_DENSE, name='drop_dense'),
    Dense(1, activation='sigmoid', name='output'),
], name='rtt_lstm_v1')

model.compile(
    optimizer=Adam(learning_rate=LR),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall'),
        AUC(name='auc_roc'),
        AUC(curve='PR', name='auc_pr'),
    ]
)

model.summary()

## Model Training

In [ ]:
class_weight_dict = {0: 1.0, 1: 8.0}
print(f'Class weights: {class_weight_dict}')

callbacks = [
    EarlyStopping(
        monitor='val_auc_pr',
        patience=PATIENCE,
        restore_best_weights=True,
        mode='max',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_auc_pr',
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        mode='max',
        verbose=1,
    ),
    ModelCheckpoint(
        model_path,
        monitor='val_auc_pr',
        save_best_only=True,
        mode='max',
        verbose=0,
    ),
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

In [ ]:
from matplotlib.ticker import MaxNLocator

# plot training and validation loss and PR-AUC side-by-side
side_by_side_metrics = [
    ('loss',   'val_loss',   'Training and Validation Loss'),
    ('auc_pr', 'val_auc_pr', 'Training and Validation PR-AUC'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (train_key, val_key, title) in zip(axes, side_by_side_metrics):
    ax.plot(history.history[train_key], label='train')
    ax.plot(history.history[val_key], label='val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves_loss_auc_pr.svg', bbox_inches='tight')
plt.show()

## Model Evaluation

In [ ]:
# Test the LSTM with the test split (20%)
y_prob = model.predict(X_test, batch_size=512, verbose=0).flatten()

pr_auc  = average_precision_score(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)
print(f'PR-AUC    (primary):   {pr_auc:.4f}')
print(f'ROC-AUC (secondary): {roc_auc:.4f}')

In [ ]:
'''
Find the best threshold value, then graph the precision-recall
 curve and the optimal threshold
'''

precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_prob)
f1s = 2 * precisions * recalls / np.clip(precisions + recalls, 1e-8, None)
opt_idx    = np.argmax(f1s[:-1])
opt_thresh = float(pr_thresholds[opt_idx])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot the PR curve
axes[0].plot(recalls, precisions, lw=1.5)
axes[0].scatter(recalls[opt_idx], precisions[opt_idx],
                color='red', zorder=5, label=f'optimal @ {opt_thresh:.2f}')

axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve (AUC={pr_auc:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot F1 vs. Threshold
axes[1].plot(pr_thresholds, f1s[:-1], lw=1.5)
axes[1].axvline(opt_thresh, color='red', linestyle='--',
                label=f'optimal @ {opt_thresh:.2f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1')
axes[1].set_title('F1 vs. Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pr_threshold_curve.svg', dpi=300)
plt.show()

final_thresh = opt_thresh
print(f'Optimal threshold: {opt_thresh:.3f}')
print(f'  Precision: {precisions[opt_idx]:.3f}')
print(f'  Recall:    {recalls[opt_idx]:.3f}')
print(f'  F1:        {f1s[opt_idx]:.3f}')

In [ ]:
# Create a confusion matrix for the optimal threshold
y_pred_opt = (y_prob >= final_thresh).astype(int)

print(classification_report(y_test, y_pred_opt, target_names=['normal', 'pre-onset']))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opt, display_labels=['normal', 'pre-onset']
)
plt.title(f'Confusion Matrix @ Threshold={final_thresh:.2f}')
plt.savefig('confusion_matrix.svg', dpi=300)
plt.show()

In [ ]:
# Compute the lead-time distribution on the test set.
lead_times_ms = []
missed = 0

run_offset = 0
for i in test_idx:
    n_windows = len(all_y[i])
    run_prob  = y_prob[run_offset : run_offset + n_windows]
    run_alert = (run_prob >= final_thresh)
    run_y     = all_y[i]
    run_offset += n_windows

    pos_windows = np.where(run_y == 1.0)[0]
    if len(pos_windows) == 0:
        continue

    # Group into contiguous segments (one per episode onset)
    seg_starts, seg_ends = [pos_windows[0]], []
    for j in range(1, len(pos_windows)):
        if pos_windows[j] > pos_windows[j-1] + 1:
            seg_ends.append(pos_windows[j-1])
            seg_starts.append(pos_windows[j])
    seg_ends.append(pos_windows[-1])

    for seg_start, seg_end in zip(seg_starts, seg_ends):
        onset_window = seg_end + 1
        if onset_window >= n_windows:
            missed += 1
            continue

        search_start = max(0, seg_start - 40)
        alerts = np.where(run_alert[search_start : onset_window])[0]
        if len(alerts) > 0:
            first_alert_win = search_start + alerts[0]
            lead_bins = onset_window - first_alert_win
            lead_times_ms.append(lead_bins * BIN_SIZE_MS)
        else:
            missed += 1

total_events = len(lead_times_ms) + missed
print(f'Total congestion onsets:    {total_events}')
print(f'Detected before onset:     {len(lead_times_ms)}  ({len(lead_times_ms)/max(total_events,1):.1%})')
print(f'Missed (no pre-alert):     {missed}')

if lead_times_ms:
    lt = np.array(lead_times_ms)
    print(f'\nLead-time stats (ms):')
    print(f'  Median:  {np.median(lt):.0f} ms')
    print(f'  Mean:    {np.mean(lt):.0f} ms')
    print(f'  p25:     {np.percentile(lt, 25):.0f} ms')
    print(f'  p75:     {np.percentile(lt, 75):.0f} ms')

    sorted_lt = np.sort(lt)
    cdf = np.arange(1, len(sorted_lt)+1) / len(sorted_lt)

    plt.figure(figsize=(8, 4))
    plt.plot(sorted_lt, cdf, lw=2)
    plt.axvline(100, color='orange', linestyle='--', label='100 ms')
    plt.axvline(200, color='red', linestyle='--', label='200 ms')
    plt.xlabel('Lead time before congestion onset (ms)')
    plt.ylabel('CDF')
    plt.title('Lead-time CDF')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('lead_time_cdf.svg', dpi=300)
    plt.show()

## Per-Run Qualitative Plots

In [ ]:
N_QUAL = min(3, len(test_idx))

run_offset = 0
prob_by_run = {}
for i in test_idx:
    n = len(all_y[i])
    prob_by_run[i] = y_prob[run_offset : run_offset + n]
    run_offset += n

fig, axes = plt.subplots(N_QUAL, 1, figsize=(14, 4 * N_QUAL))
if N_QUAL == 1:
    axes = [axes]

for ax, run_i in zip(axes, test_idx[:N_QUAL]):
    probs  = prob_by_run[run_i]
    labels = all_y[run_i]
    t_s    = np.arange(len(probs)) * (BIN_SIZE_MS * STRIDE) / 1000.0

    ax.plot(t_s, probs, lw=1.2, color='steelblue', label='P(congestion-imminent)')
    ax.fill_between(t_s, 0, 1, where=(labels == 1),
                    color='red', alpha=0.15, label='Ground-Truth Label')
    ax.axhline(final_thresh, color='orange', linestyle='--', lw=1,
               label=f'Threshold={final_thresh:.2f}')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('P(congestion-imminent)')
    ax.set_title(f'Test run {run_i}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('qualitative_test.svg', dpi=300)
plt.show()

## Export Results

In [ ]:
model.save(model_path)
print(f'Model saved to {model_path}')

metadata = {
    'feature_cols':       FEATURE_COLS,
    'seq_len':            SEQ_LEN,
    'bin_size_ms':        BIN_SIZE_MS,
    'n_features':         N_FEATURES,
    'threshold':          float(final_thresh),
    'pr_auc':             float(pr_auc),
    'roc_auc':            float(roc_auc),
    'best_f1':            float(f1s[opt_idx]),
    'label_method':       'kernel (ca_state >= 3 | is_timeout >= 1)',
    'threshold_K':        K,
    'n_pre_onset_rtts':   N_PRE_ONSET_RTTS,
    'normalization':      'per_connection_min_rtt',
}

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata saved to {meta_path}')
print(json.dumps(metadata, indent=2))